In [ ]:

from cellpose import models, utils
from skimage.exposure import equalize_adapthist
from skimage.morphology import white_tophat, closing, disk, label
from skimage.measure import regionprops
from scipy.ndimage import gaussian_filter
import numpy as np
import matplotlib.pyplot as plt
from import_images import encontrar_imagens_tiff, carregar_imagem_por_indice
import os


class ProcessadorDeImagens:
    def __init__(self, base_dir, modelo='nuclei', canais=[0, 0]):
        self.image_paths = encontrar_imagens_tiff(base_dir)
        self.modelo = models.Cellpose(model_type=modelo)
        self.canais = canais
        cellviability_dir = base_dir
        while os.path.basename(cellviability_dir) != "CellViability":
            novo_dir = os.path.dirname(cellviability_dir)
            if novo_dir == cellviability_dir:  # chegou na raiz e não encontrou
                raise FileNotFoundError("Diretório 'CellViability' não encontrado na hierarquia acima.")
            cellviability_dir = novo_dir

        # Define pasta de saída em CellViability/resultados/cellpose
        self.output_dir = os.path.join(cellviability_dir, "resultados/cellpose")

        os.makedirs(self.output_dir, exist_ok=True)

    def preprocessar_imagem(self, imagem):
        imagem = gaussian_filter(imagem, sigma=1)
        imagem = white_tophat(imagem, footprint=disk(15))
        imagem = equalize_adapthist(imagem, clip_limit=0.4)
        imagem = (imagem - np.min(imagem)) / (np.max(imagem) - np.min(imagem))
        imagem = closing(imagem, disk(3))
        return imagem

    def filtrar_objetos(self, mascara, imagem_original, limiar_circularidade=0.75, limiar_area=80, fator_intensidade=1.2):
        nova_mascara = np.zeros_like(mascara)
        props = regionprops(label(mascara), intensity_image=imagem_original)

        media_imagem = np.mean(imagem_original)
        index = 1
        for prop in props:
            if prop.perimeter == 0:
                continue

            circularidade = (4 * np.pi * prop.area) / (prop.perimeter ** 2)
            intensidade_media = prop.mean_intensity
            area = prop.area

            if (
                circularidade >= limiar_circularidade and
                area >= limiar_area and
                intensidade_media > media_imagem * fator_intensidade
            ):
                nova_mascara[label(mascara) == prop.label] = index
                index += 1

        return nova_mascara

    def carregar_e_processar(self, indice):
        imagem_original = carregar_imagem_por_indice(self.image_paths, indice)
        if imagem_original is None:
            print("Não foi possível carregar a imagem.")
            return None, None

        print("Pré-processando imagem...")
        imagem = self.preprocessar_imagem(imagem_original)

        print("Segmentando com Cellpose...")
        masks, flows, styles, diams = self.modelo.eval(
            imagem,
            channels=self.canais,
            flow_threshold=1,
            cellprob_threshold=-0.1,
            min_size=300
        )

        print("Filtrando objetos...")
        masks_filtradas = self.filtrar_objetos(
            mascara=masks,
            imagem_original=imagem_original,
            limiar_circularidade=0.75,
            limiar_area=80,
            fator_intensidade=1.2
        )

        n_objetos = len(np.unique(masks_filtradas)) - 1
        print(f"{n_objetos} objetos circulares mantidos")

        outlines = utils.outlines_list(masks_filtradas)
        nome_arquivo = os.path.basename(self.image_paths[indice])
        nome_base = os.path.splitext(nome_arquivo)[0]
        caminho_saida = os.path.join(self.output_dir, f"{nome_base}_segmentado.png")

        fig, ax = plt.subplots(figsize=(6, 6))
        ax.imshow(imagem_original, cmap='gray')
        for o in outlines:
            ax.plot(o[:, 0], o[:, 1], color='red', linewidth=0.5)

        ax.set_title(f"{nome_base} - {n_objetos} objetos circulares")
        ax.axis('off')
        plt.savefig(caminho_saida, dpi=300, bbox_inches='tight')
        plt.close()

        print(f"Segmentação salva em: {caminho_saida}")
        return masks_filtradas, flows

if __name__ == "__main__":
    base_dir="/home/kayllany.oliveira/remote-repos/CellViability/data/17 Resultados HTS SLEV/SLEV HTS TargetMol/S_1_R1[10270]/2022-04-11T161022Z[10923]"
    proc = ProcessadorDeImagens(base_dir=base_dir, modelo='nuclei')
    image_paths = proc.image_paths

    print(f"Total de imagens encontradas: {len(image_paths)}")
    for i, path in image_paths.items():
        masks, flows = proc.carregar_e_processar(i)



In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # Força uso da CPU, desativa GPU

import cv2
import numpy as np
import tifffile
from stardist.models import StarDist2D
from csbdeep.utils import normalize
from skimage.morphology import remove_small_objects
from skimage.measure import label, regionprops

class SegmentadorStarDist:
    def __init__(self, base_dir, resultados_dir):
        self.base_dir = base_dir
        self.resultados_dir = resultados_dir
        os.makedirs(self.resultados_dir, exist_ok=True)

        self.modelo_stardist = StarDist2D(None,
                                          name="2D_versatile_fluo",
                                          basedir="/home/kayllany.oliveira/remote-repos/CellViability")

    def carregar_imagens(self):
        imagens = []
        for nome_arquivo in os.listdir(self.base_dir):
            if nome_arquivo.lower().endswith((".tif", ".tiff")):
                caminho = os.path.join(self.base_dir, nome_arquivo)
                imagens.append(caminho)
        return imagens

    def segmentar(self, imagem):
        if imagem.ndim == 2:
            imagem = np.expand_dims(imagem, axis=-1)
        elif imagem.ndim == 3 and imagem.shape[-1] not in [1, 3]:
            raise ValueError(f"Imagem com shape inválido para StarDist: {imagem.shape}")

        labels, _ = self.modelo_stardist.predict_instances(imagem)
        labels = self.posprocessar_labels(labels)
        return labels

    def posprocessar_labels(self, labels, min_area=50):
        labels = remove_small_objects(labels, min_size=min_area)
        return labels

    def processar_todas_as_imagens(self):
        print("[DEBUG] Iniciando processamento de imagens...")
        imagens = self.carregar_imagens()
        print(f"[DEBUG] Imagens encontradas: {len(imagens)}")
        if not imagens:
            print("[AVISO] Nenhuma imagem .tif/.tiff encontrada no diretório base.")
            return

        for caminho_imagem in imagens:
            print(f"[DEBUG] Processando: {caminho_imagem}")
            try:
                img = tifffile.imread(caminho_imagem)
                img_pre = preprocessar_imagem_para_fluorescencia(img)
                labels = self.segmentar(img_pre)

                nome_arquivo = os.path.splitext(os.path.basename(caminho_imagem))[0]
                caminho_saida = os.path.join(self.resultados_dir, f"{nome_arquivo}_stardist_contorno.png")
                print(f"[DEBUG] Salvando resultado em: {caminho_saida}")

                sucesso = salvar_contorno_com_texto(img, labels, caminho_saida)
                if sucesso:
                    print(f"[DEBUG] Imagem salva com sucesso: {caminho_saida}")
                else:
                    print(f"[ERRO] Falha ao salvar a imagem: {caminho_saida}")

            except Exception as e:
                print(f"[ERRO] ao processar {caminho_imagem}: {e}")

def preprocessar_imagem_para_fluorescencia(imagem):
    imagem = normalize(imagem.astype(np.float32), 1, 99.8, clip=True)
    imagem = (imagem * 255).astype(np.uint8)
    #imagem = cv2.equalizeHist(imagem)  # <-- comentado para evitar alteração forte
    imagem = cv2.medianBlur(imagem, 3)
    return imagem

def salvar_contorno_com_texto(imagem_original, labels, caminho_saida):
    try:
        contorno = np.zeros_like(imagem_original, dtype=np.uint8)
        for region in regionprops(label(labels)):
            for coord in region.coords:
                contorno[coord[0], coord[1]] = 255

        if imagem_original.ndim == 3:
            imagem_original = imagem_original[..., 0]

        # Inverter imagem para que núcleos fiquem claros e fundo preto (se necessário)
        imagem_invertida = 255 - imagem_original

        resultado_rgb = cv2.merge([imagem_invertida]*3)
        resultado_rgb[contorno == 255] = [255, 0, 0]  # vermelho

        num_celulas = len(np.unique(labels)) - 1
        texto = f"CELL: {num_celulas}"
        fonte = cv2.FONT_HERSHEY_SIMPLEX
        cv2.putText(resultado_rgb, texto, (10, 25), fonte, 1, (0, 255, 0), 2, cv2.LINE_AA)

        cv2.imwrite(caminho_saida, resultado_rgb)
        return True
    except Exception as e:
        print(f"[ERRO] ao salvar imagem: {e}")
        return False

if __name__ == "__main__":
    base_dir = "/home/kayllany.oliveira/remote-repos/CellViability/data/17 Resultados HTS SLEV/SLEV HTS TargetMol/S_1_R1[10270]/2022-04-11T161022Z[10923]"
    resultados_dir = "/home/kayllany.oliveira/remote-repos/CellViability/resultados/stardist"

    segmentador = SegmentadorStarDist(base_dir, resultados_dir)
    segmentador.processar_todas_as_imagens()


In [1]:
from cellpose import models, utils
from skimage.exposure import equalize_adapthist
from skimage.morphology import white_tophat, closing, disk, label
from skimage.measure import regionprops
from scipy.ndimage import gaussian_filter
import numpy as np
import matplotlib.pyplot as plt
from import_images import encontrar_imagens_tiff, carregar_imagem_por_indice
import os


class ProcessadorDeImagens:
    def __init__(self, base_dir, modelo='nuclei', canais=[0, 0]):
        self.image_paths = encontrar_imagens_tiff(base_dir)
        self.modelo = models.Cellpose(model_type=modelo)
        self.canais = canais

        cellviability_dir = base_dir
        while os.path.basename(cellviability_dir) != "CellViability":
            novo_dir = os.path.dirname(cellviability_dir)
            if novo_dir == cellviability_dir:  # chegou na raiz e não encontrou
                raise FileNotFoundError("Diretório 'CellViability' não encontrado na hierarquia acima.")
            cellviability_dir = novo_dir

        # Define pasta de saída em CellViability/resultados/cellpose
        self.output_dir = os.path.join(cellviability_dir, "resultados/cellpose")
        os.makedirs(self.output_dir, exist_ok=True)

    def preprocessar_imagem(self, imagem):
        # Pré-processamento com parâmetros ajustados para núcleos pequenos
        imagem = gaussian_filter(imagem, sigma=1)
        imagem = white_tophat(imagem, footprint=disk(7))  # menor disco para preservar pequenos detalhes
        imagem = equalize_adapthist(imagem, clip_limit=0.4)
        imagem = (imagem - np.min(imagem)) / (np.max(imagem) - np.min(imagem))
        imagem = closing(imagem, disk(2))  # fechamento leve
        return imagem

    def filtrar_objetos(self, mascara, imagem_original,
                        limiar_circularidade=0.6,
                        limiar_area=40,
                        fator_intensidade=1.0):
        nova_mascara = np.zeros_like(mascara)
        props = regionprops(label(mascara), intensity_image=imagem_original)

        media_imagem = np.mean(imagem_original)
        index = 1
        for prop in props:
            if prop.perimeter == 0:
                continue

            circularidade = (4 * np.pi * prop.area) / (prop.perimeter ** 2)
            intensidade_media = prop.mean_intensity
            area = prop.area

            if (
                circularidade >= limiar_circularidade and
                area >= limiar_area and
                intensidade_media > media_imagem * fator_intensidade
            ):
                nova_mascara[label(mascara) == prop.label] = index
                index += 1

        return nova_mascara

    def carregar_e_processar(self, indice):
        imagem_original = carregar_imagem_por_indice(self.image_paths, indice)
        if imagem_original is None:
            print("Não foi possível carregar a imagem.")
            return None, None

        print(f"Processando imagem {indice + 1}/{len(self.image_paths)}: {os.path.basename(self.image_paths[indice])}")

        print("Pré-processando imagem...")
        imagem = self.preprocessar_imagem(imagem_original)

        print("Segmentando com Cellpose...")
        masks, flows, styles, diams = self.modelo.eval(
            imagem,
            channels=self.canais,
            flow_threshold=0.6,         # menor threshold para pegar mais núcleos pequenos
            cellprob_threshold=0.0,     # mais permissivo para incluir núcleos menos evidentes
            min_size=30                 # permitir núcleos menores
        )

        print("Filtrando objetos...")
        masks_filtradas = self.filtrar_objetos(
            mascara=masks,
            imagem_original=imagem_original,
            limiar_circularidade=0.6,
            limiar_area=40,
            fator_intensidade=1.0
        )

        n_objetos = len(np.unique(masks_filtradas)) - 1
        print(f"{n_objetos} objetos circulares mantidos")

        outlines = utils.outlines_list(masks_filtradas)
        nome_arquivo = os.path.basename(self.image_paths[indice])
        nome_base = os.path.splitext(nome_arquivo)[0]
        caminho_saida = os.path.join(self.output_dir, f"{nome_base}_segmentado.png")

        fig, ax = plt.subplots(figsize=(6, 6))
        ax.imshow(imagem_original, cmap='gray')
        for o in outlines:
            ax.plot(o[:, 0], o[:, 1], color='red', linewidth=0.5)

        ax.set_title(f"{nome_base} - {n_objetos} CELLS")
        ax.axis('off')
        plt.savefig(caminho_saida, dpi=300, bbox_inches='tight')
        plt.close()

        print(f"Segmentação salva em: {caminho_saida}")
        return masks_filtradas, flows


if __name__ == "__main__":
    base_dir = "/home/kayllany.oliveira/remote-repos/CellViability/data/17 Resultados HTS SLEV/SLEV HTS TargetMol/S_1_R1[10270]/2022-04-11T161022Z[10923]"
    proc = ProcessadorDeImagens(base_dir=base_dir, modelo='nuclei')
    image_paths = proc.image_paths

    print(f"Total de imagens encontradas: {len(image_paths)}")
    for i in range(len(image_paths)):
        masks, flows = proc.carregar_e_processar(i)


/home/kayllany.oliveira/remote-repos/CellViability/data/17 Resultados HTS SLEV/SLEV HTS TargetMol/S_1_R1[10270]/2022-04-11T161022Z[10923]/008012-1-001001001.tif
/home/kayllany.oliveira/remote-repos/CellViability/data/17 Resultados HTS SLEV/SLEV HTS TargetMol/S_1_R1[10270]/2022-04-11T161022Z[10923]/011008-1-001001001.tif
/home/kayllany.oliveira/remote-repos/CellViability/data/17 Resultados HTS SLEV/SLEV HTS TargetMol/S_1_R1[10270]/2022-04-11T161022Z[10923]/016020-1-001001001.tif
/home/kayllany.oliveira/remote-repos/CellViability/data/17 Resultados HTS SLEV/SLEV HTS TargetMol/S_1_R1[10270]/2022-04-11T161022Z[10923]/005020-1-001001001.tif
/home/kayllany.oliveira/remote-repos/CellViability/data/17 Resultados HTS SLEV/SLEV HTS TargetMol/S_1_R1[10270]/2022-04-11T161022Z[10923]/016018-1-001001001.tif
/home/kayllany.oliveira/remote-repos/CellViability/data/17 Resultados HTS SLEV/SLEV HTS TargetMol/S_1_R1[10270]/2022-04-11T161022Z[10923]/015009-1-001001001.tif
/home/kayllany.oliveira/remote-rep